# CI/CD 质量门禁与在线监控

前置知识：本章 Notebook 2-5

本节目标：将评估集成到开发流程中——用 DeepEval 编写 pytest 风格的评估测试，模拟 CI 质量门禁，并掌握在线监控的核心指标。

## 一、为什么 RAG 需要 CI/CD

RAG 系统有多个可变组件：索引策略、embedding 模型、检索参数、prompt 模板、生成模型。任何一个组件的改动都可能影响最终质量。

如果没有自动化评估：
- 改了 prompt 以为变好了，但某类问题变差了 → **回归风险**
- 换了 embedding 模型，不知道对哪些场景有影响 → **变更盲区**
- 上线后才发现问题 → **线上事故**

CI/CD 质量门禁的核心思想：**每次改动前，自动跑评估集，得分不达标就不允许合并代码。**

## 二、质量门禁设计

### 门禁阈值
| 指标 | 最低阈值 | 含义 |
|------|---------|------|
| Faithfulness | ≥ 0.80 | 答案必须有据可查 |
| Answer Relevancy | ≥ 0.85 | 答案必须切题 |
| Context Recall | ≥ 0.70 | 关键信息必须被检索到 |
| 回归不退化 | Δ ≤ -0.05 | 任何指标下降不超过 5% |

### 门禁流程
```
改动代码 → 自动触发评估 → 对比 baseline → 判断是否达标 → 通过/拒绝
```

### 版本化
全链路都需要版本化：
- 索引参数版本
- Embedding 模型版本
- Prompt 模板版本
- 生成模型版本
- 评估集版本

In [ ]:
import os
import json
import time
import numpy as np
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-small-zh-v1.5', cache_dir='./models')

In [ ]:
import re
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embedding = HuggingFaceEmbeddings(model_name=model_dir)
api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(model="glm-4-flash", temperature=0.0, api_key=api_key)

pdf_path = "../3. 索引阶段/data/pumpkin_book.pdf"
persist_dir = "./chroma_db"

def clean_text(text):
    text = re.sub(r'→_→\n.*?←_←', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_vectorstore(pdf_path, embedding, persist_directory="./chroma_db"):
    """构建或加载向量库，已有则复用"""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"发现已存在的向量库: {persist_directory}，正在加载...")
        try:
            vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            count = vectorstore._collection.count()
            print(f"✅ 加载成功！共 {count} 个文档块")
            return vectorstore
        except Exception as e:
            print(f"⚠️ 加载失败 ({e})，将重新构建...")
    print("开始构建向量库...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_pages = loader.load()
    data_pages = pdf_pages[13:-13]
    for page in data_pages:
        page.page_content = clean_text(page.page_content)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(data_pages)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory)
    print(f"✅ 向量库构建完成并保存至 {persist_directory}，共 {len(splits)} 个文档块")
    return vectorstore

vectorstore = build_vectorstore(pdf_path, embedding, persist_dir)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

print("RAG Pipeline 构建完成")

## 三、用 DeepEval 编写评估测试

[DeepEval](https://docs.confident-ai.com/) 是一个 pytest 风格的 LLM 评估框架，可以像写单元测试一样写评估用例。

安装：`pip install deepeval`

In [ ]:
# DeepEval 概念演示：pytest 风格的评估测试
# 注：实际项目中推荐直接使用 deepeval 库

class LLMTestCase:
    """模拟 DeepEval 的 LLMTestCase"""
    def __init__(self, input_text, actual_output, expected_output=None, retrieval_context=None):
        self.input = input_text
        self.actual_output = actual_output
        self.expected_output = expected_output
        self.retrieval_context = retrieval_context or []

class FaithfulnessMetric:
    """模拟 Faithfulness 评估指标"""
    def __init__(self, threshold=0.8):
        self.threshold = threshold
        self.score = None
        self.reason = None

    def measure(self, test_case):
        context = "\n\n".join(test_case.retrieval_context[:3])
        judge_prompt = ChatPromptTemplate.from_template(
            "判断以下回答是否基于给定上下文。返回 0.0-1.0 的分数。\n\n"
            "上下文：{context}\n回答：{answer}\n\n"
            "只返回一个小数，如 0.85："
        )
        chain = judge_prompt | llm | StrOutputParser()
        resp = chain.invoke({"context": context, "answer": test_case.actual_output})
        try:
            self.score = float(resp.strip())
            self.score = max(0.0, min(1.0, self.score))
        except ValueError:
            self.score = 0.5
        self.reason = f"Faithfulness score: {self.score}"
        return self.score

    def is_successful(self):
        return self.score is not None and self.score >= self.threshold

class AnswerRelevancyMetric:
    """模拟 Answer Relevancy 评估指标"""
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.score = None
        self.reason = None

    def measure(self, test_case):
        judge_prompt = ChatPromptTemplate.from_template(
            "判断以下回答是否切题回答了问题。返回 0.0-1.0 的分数。\n\n"
            "问题：{question}\n回答：{answer}\n\n"
            "只返回一个小数，如 0.85："
        )
        chain = judge_prompt | llm | StrOutputParser()
        resp = chain.invoke({"question": test_case.input, "answer": test_case.actual_output})
        try:
            self.score = float(resp.strip())
            self.score = max(0.0, min(1.0, self.score))
        except ValueError:
            self.score = 0.5
        self.reason = f"Answer Relevancy score: {self.score}"
        return self.score

    def is_successful(self):
        return self.score is not None and self.score >= self.threshold

def assert_test(test_case, metrics):
    """模拟 DeepEval 的 assert_test"""
    results = []
    for metric in metrics:
        score = metric.measure(test_case)
        passed = metric.is_successful()
        results.append({
            "metric": type(metric).__name__,
            "score": score,
            "threshold": metric.threshold,
            "passed": passed,
        })
    return results

print("评估测试框架定义完成")

In [ ]:
rag_prompt_tpl = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题。如果上下文中没有相关信息，请说'根据已有资料无法回答'。\n\n"
    "上下文：\n{context}\n\n问题：{question}\n\n回答："
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt_tpl | llm | StrOutputParser()
)

test_questions = [
    {"question": "什么是信息增益？", "ground_truth": "信息增益是指在得知某个特征的信息后，信息不确定性减少的程度。"},
    {"question": "什么是基尼指数？", "ground_truth": "基尼指数是度量数据集纯度的一种指标。"},
    {"question": "过拟合和欠拟合的区别？", "ground_truth": "过拟合是训练好测试差，欠拟合是都差。"},
]

print("运行 v1 评估测试...")
print("=" * 70)

v1_results = []
for q in test_questions:
    docs = retriever.invoke(q["question"])
    answer = rag_chain.invoke(q["question"])

    test_case = LLMTestCase(
        input_text=q["question"],
        actual_output=answer,
        expected_output=q["ground_truth"],
        retrieval_context=[doc.page_content for doc in docs]
    )

    metrics = [FaithfulnessMetric(threshold=0.8), AnswerRelevancyMetric(threshold=0.85)]
    results = assert_test(test_case, metrics)
    v1_results.append({"question": q["question"], "answer": answer, "results": results})

    status = "PASS" if all(r["passed"] for r in results) else "FAIL"
    print(f"\n[{status}] {q['question']}")
    for r in results:
        flag = "✓" if r["passed"] else "✗"
        print(f"  {flag} {r['metric']}: {r['score']:.2f} (阈值: {r['threshold']})")
    time.sleep(1)

## 四、模拟 CI 流程：Prompt 变更对比

在实际 CI 中，每次 prompt 变更都需要跑评估集，对比变更前后的得分。下面我们模拟这个流程：
1. v1 prompt → 跑评估 → 记录基线得分
2. 修改 prompt → v2 prompt → 跑评估 → 记录新得分
3. 对比两个版本的得分，判断是否可以合并

In [ ]:
v2_prompt = ChatPromptTemplate.from_template(
    "你是一个机器学习领域的助教。请严格根据以下参考资料回答学生的问题。\n"
    "如果参考资料中没有相关信息，请明确告知学生'根据现有资料无法回答该问题'。\n"
    "回答时请引用具体的知识点，保持简洁准确。\n\n"
    "参考资料：\n{context}\n\n"
    "学生问题：{question}\n\n"
    "回答："
)

rag_chain_v2 = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | v2_prompt | llm | StrOutputParser()
)

print("运行 v2 评估测试...")
print("=" * 70)

v2_results = []
for q in test_questions:
    docs = retriever.invoke(q["question"])
    answer = rag_chain_v2.invoke(q["question"])

    test_case = LLMTestCase(
        input_text=q["question"],
        actual_output=answer,
        expected_output=q["ground_truth"],
        retrieval_context=[doc.page_content for doc in docs]
    )

    metrics = [FaithfulnessMetric(threshold=0.8), AnswerRelevancyMetric(threshold=0.85)]
    results = assert_test(test_case, metrics)
    v2_results.append({"question": q["question"], "answer": answer, "results": results})

    status = "PASS" if all(r["passed"] for r in results) else "FAIL"
    print(f"\n[{status}] {q['question']}")
    for r in results:
        flag = "✓" if r["passed"] else "✗"
        print(f"  {flag} {r['metric']}: {r['score']:.2f} (阈值: {r['threshold']})")
    time.sleep(1)

In [ ]:
print("版本对比报告")
print("=" * 70)
print(f"{'问题':<20} {'指标':<20} {'v1':>6} {'v2':>6} {'变化':>8} {'判定':>6}")
print("-" * 70)

gate_passed = True
for i in range(len(test_questions)):
    q = test_questions[i]["question"][:18]
    for j in range(len(v1_results[i]["results"])):
        metric = v1_results[i]["results"][j]["metric"]
        s1 = v1_results[i]["results"][j]["score"]
        s2 = v2_results[i]["results"][j]["score"]
        delta = s2 - s1

        if delta < -0.05:
            verdict = "退化"
            gate_passed = False
        elif delta > 0.05:
            verdict = "提升"
        else:
            verdict = "持平"

        print(f"{q:<20} {metric:<20} {s1:>6.2f} {s2:>6.2f} {delta:>+8.2f} {verdict:>6}")

print("\n" + "=" * 70)
if gate_passed:
    print("质量门禁：PASS - v2 prompt 可以合并")
else:
    print("质量门禁：FAIL - v2 prompt 存在退化，需要修复后重新提交")

## 五、在线监控

质量门禁保证了上线前的质量，但上线后仍需持续监控。核心在线指标：

| 指标 | 含义 | 采集方式 | 告警阈值 |
|------|------|---------|----------|
| 追问率 | 用户在一次问答后继续追问的比例 | 会话日志 | > 30% |
| 拒答率 | 系统返回"无法回答"的比例 | 响应日志 | > 15% |
| 延迟 P95 | 95% 请求的端到端延迟 | 系统监控 | > 5s |
| Token 成本 | 每次问答的平均 token 消耗 | API 日志 | 超预算 |
| 用户反馈 | 用户点赞/点踩比例 | 前端埋点 | 踩率 > 20% |

In [ ]:
import datetime

class OnlineMetricsCollector:
    """在线指标采集器（演示用）"""

    def __init__(self):
        self.records = []

    def record(self, question, answer, latency_ms, token_count, is_refusal=False):
        self.records.append({
            "timestamp": datetime.datetime.now().isoformat(),
            "question": question,
            "answer_length": len(answer),
            "latency_ms": latency_ms,
            "token_count": token_count,
            "is_refusal": is_refusal,
        })

    def report(self):
        if not self.records:
            return "无数据"

        latencies = [r["latency_ms"] for r in self.records]
        tokens = [r["token_count"] for r in self.records]
        refusals = sum(1 for r in self.records if r["is_refusal"])

        return {
            "total_queries": len(self.records),
            "avg_latency_ms": np.mean(latencies),
            "p95_latency_ms": np.percentile(latencies, 95),
            "avg_tokens": np.mean(tokens),
            "refusal_rate": refusals / len(self.records),
        }

collector = OnlineMetricsCollector()

for q in test_questions:
    start = time.time()
    answer = rag_chain.invoke(q["question"])
    latency = (time.time() - start) * 1000
    is_refusal = "无法回答" in answer
    collector.record(q["question"], answer, latency, len(answer) * 2, is_refusal)

report = collector.report()
print("在线监控报告")
print("=" * 40)
for k, v in report.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")

## 六、全链路版本化

生产环境中，每次评估都需要记录完整的版本信息，确保可复现：

```python
version_info = {
    "eval_date": "2026-03-15",
    "eval_set_version": "v1.0",
    "index_config": {
        "chunk_size": 500,
        "chunk_overlap": 50,
        "embedding_model": "BAAI/bge-small-zh-v1.5"
    },
    "retrieval_config": {
        "top_k": 5,
        "search_type": "similarity"
    },
    "generation_config": {
        "model": "glm-4-flash",
        "temperature": 0.0,
        "prompt_version": "v2"
    }
}
```

将版本信息和评估结果一起保存，方便后续对比和回溯。

## 七、GitHub Actions 集成示例

以下是一个 CI 配置示例，在每次 PR 时自动运行评估：

```yaml
# .github/workflows/rag-eval.yml
name: RAG Evaluation
on:
  pull_request:
    paths:
      - 'prompts/**'
      - 'config/**'

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      - run: pip install -r requirements.txt
      - run: python -m pytest tests/test_rag_eval.py --tb=short
        env:
          ZHIPUAI_API_KEY: ${{ secrets.ZHIPUAI_API_KEY }}
```

注意：这里只展示配置结构，实际 CI 还需要考虑向量库的缓存、评估数据的版本管理等。

## 八、小结

### 关键要点
1. **自动化评估**：每次改动都跑评估，用数据说话
2. **质量门禁**：设定阈值，不达标不合并
3. **版本对比**：对比变更前后的得分，量化改动效果
4. **在线监控**：上线后持续追踪，及时发现退化
5. **全链路版本化**：确保每次评估可复现

### 实践建议
1. **先建基线**：在引入 CI 前，先跑一轮完整评估建立基线
2. **渐进式引入**：先监控不拦截，稳定后再开启门禁
3. **合理设阈值**：阈值太严会阻塞开发，太松没有意义
4. **保留历史**：所有评估记录都保存，用于趋势分析

### 参考文献
- [DeepEval 官方文档](https://docs.confident-ai.com/)
- [Continuous Evaluation for LLM Applications](https://www.confident-ai.com/blog)